# 🎵 K-Pop Dance Analyzer
### วิเคราะห์และเปรียบเทียบท่าเต้นของคุณกับศิลปิน K-pop

**วิธีใช้งาน:**
1. ติดตั้ง dependencies (Cell 1)
2. ใส่ YouTube URL ของ K-pop MV ที่ต้องการ (Cell 3)
3. ใส่ path ของวิดีโอตัวเองที่เต้นแล้ว (Cell 3)
4. Run ทุก cell ตามลำดับ

---
**ข้อแนะนำสำหรับวิดีโอของตัวเอง:**
- 🎥 ถ่ายให้เห็นทั้งร่างกาย head to toe
- 💡 แสงสว่างเพียงพอ ไม่ย้อนแสง
- 📐 ถ่ายจากมุมตรง ไม่เอียงมาก
- ⏱️ ความยาว 15 วินาที - 2 นาที เหมาะที่สุด

In [1]:
# Cell 1: ติดตั้ง dependencies
# (รันครั้งแรกครั้งเดียว)
import subprocess, sys
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "-r", "requirements.txt",
    "--break-system-packages", "-q"
])
print("✅ ติดตั้ง dependencies เสร็จแล้ว!")

✅ ติดตั้ง dependencies เสร็จแล้ว!


In [2]:
# Cell 2: Import libraries
import os, importlib
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display, Image, Video, HTML
import importlib, reference_processor
importlib.reload(reference_processor)

# reload เสมอเพื่อรับโค้ดใหม่จากไฟล์
import reference_processor, dance_comparator
importlib.reload(reference_processor)
importlib.reload(dance_comparator)

from reference_processor import (
    download_youtube_video,
    extract_pose_from_video,
    preview_person_selection,
    create_labeled_video,
    create_person_clip,
    save_pose_data,
    load_pose_data,
)
from dance_comparator import compare_dance, quick_compare, grade_score

print('✅ Import สำเร็จ! พร้อมใช้งาน')

✅ Import สำเร็จ! พร้อมใช้งาน


## ⚙️ ตั้งค่า
แก้ไข cell นี้ตามความต้องการ

In [3]:
# Cell 3: ตั้งค่าหลัก

# === ตั้งค่า Reference Video (K-pop Artist) ===
YOUTUBE_URL = "https://www.youtube.com/watch?v=Ba-y6OgrC70"   # 👈 ใส่ YouTube URL ที่นี่
START_TIME  = "0:00"
END_TIME    = "2:44"

# === เลือกคนที่ต้องการ track ===
PERSON_INDEX = None   # 👈 แก้ตรงนี้หลังดู preview

# === วิดีโอของคุณ ===
# ใส่ path ไฟล์: USER_VIDEO = "my_dance.mp4"
# หรือใส่ YouTube URL: USER_VIDEO = "https://youtu.be/xxxxx"
USER_VIDEO       = "https://youtu.be/ottSiNd8YEA?si=ZUKvnNNl3j0ICnjB"   # 👈 ใส่ path หรือ YouTube URL
USER_START_TIME  = "0:00"  # เช่น "0:30"  หรือ None = ตั้งแต่ต้น
USER_END_TIME    = "2:44"  # เช่น "1:30"  หรือ None = จนจบ

# === ตั้งค่าอื่นๆ ===
SAMPLE_FPS = 10
OUTPUT_DIR = "results"

print(f'Reference: {YOUTUBE_URL}  [{START_TIME} → {END_TIME}]')
print(f'Person index: {PERSON_INDEX}')
print(f'วิดีโอของคุณ: {USER_VIDEO}  [{USER_START_TIME} → {USER_END_TIME}]')

Reference: https://www.youtube.com/watch?v=Ba-y6OgrC70  [0:00 → 2:44]
Person index: None
วิดีโอของคุณ: https://youtu.be/ottSiNd8YEA?si=ZUKvnNNl3j0ICnjB  [0:00 → 2:44]


## 📥 Step 1: ดาวน์โหลด Reference Video

In [4]:
# Cell 4a: ดาวน์โหลดวิดีโอ reference จาก YouTube
import importlib, reference_processor
importlib.reload(reference_processor)
from reference_processor import download_youtube_video

ref_video_path = download_youtube_video(
    url=YOUTUBE_URL,
    output_dir=os.path.join(OUTPUT_DIR, "reference_videos"),
    start_time=START_TIME,
    end_time=END_TIME,
)
print(f'✅ ดาวน์โหลดสำเร็จ: {ref_video_path}')

  URL (clean): https://www.youtube.com/watch?v=Ba-y6OgrC70
กำลังดาวน์โหลดวิดีโอจาก: https://www.youtube.com/watch?v=Ba-y6OgrC70
ช่วงเวลา: 0:00 → 2:44
[youtube] Extracting URL: https://www.youtube.com/watch?v=Ba-y6OgrC70
[youtube] Ba-y6OgrC70: Downloading webpage
[youtube] Ba-y6OgrC70: Downloading android vr player API JSON
[youtube] Ba-y6OgrC70: Downloading player 1bb6ee63-main
[youtube] [jsc:deno] Solving JS challenges using deno
[youtube] Ba-y6OgrC70: Downloading m3u8 information
[info] Ba-y6OgrC70: Downloading 1 format(s): 137+140
[info] Ba-y6OgrC70: Downloading 1 time ranges: 0.0-164.0
[download] results/reference_videos/[Mirrored] KARINA(카리나) - UP(업) 1인 커버댄스 ㅣ1인안무 거울모드.mp4 has already been downloaded
ดาวน์โหลดสำเร็จ: results/reference_videos/[Mirrored] KARINA(카리나) - UP(업) 1인 커버댄스 ㅣ1인안무 거울모드.mp4
✅ ดาวน์โหลดสำเร็จ: results/reference_videos/[Mirrored] KARINA(카리나) - UP(업) 1인 커버댄스 ㅣ1인안무 거울모드.mp4


### 👥 Cell 4b: ดูว่ามีกี่คนในวิดีโอ (สำหรับวิดีโอที่มีนักเต้นหลายคน)
รัน cell นี้เพื่อดูภาพ preview พร้อมกรอบสีรอบแต่ละคน
แล้วนำหมายเลขที่ต้องการกลับไปใส่ `PERSON_INDEX` ใน Cell 3

In [ ]:
# Cell 4b: คลิกเลือกคนที่ต้องการ track
# หน้าต่างจะเปิดขึ้นมา → คลิกที่คนที่ต้องการ → ปิดหน้าต่าง
os.makedirs(OUTPUT_DIR, exist_ok=True)
preview_path = os.path.join(OUTPUT_DIR, 'person_preview.png')

selected_index = preview_person_selection(
    video_path=ref_video_path,
    save_path=preview_path,
)

# อัปเดต PERSON_INDEX อัตโนมัติ
PERSON_INDEX = selected_index
print(f'✅ PERSON_INDEX ถูกตั้งเป็น {PERSON_INDEX} แล้ว รัน Cell 5 ได้เลยครับ')

In [ ]:
# Cell 4c: ดูวิดีโอที่มีกรอบหมายเลขทุกคน → ดูว่า PERSON_INDEX ไหนคือคนที่ต้องการ
import importlib, reference_processor
importlib.reload(reference_processor)
from reference_processor import create_labeled_video
from IPython.display import Video

labeled_path = os.path.join(OUTPUT_DIR, "labeled_preview.mp4")
create_labeled_video(video_path=ref_video_path, output_path=labeled_path)

print(f"✅ ดูวิดีโอแล้วจด PERSON_INDEX ของคนที่ต้องการ")
display(Video(labeled_path, width=800, embed=True))

## 🦴 Step 2: Extract Pose จาก Reference Video

In [5]:
# Cell 5: Extract pose keypoints จาก reference
# (ถ้ามีหลายคน ให้แก้ PERSON_INDEX ใน Cell 3 ก่อนรัน cell นี้)
print('กำลัง extract pose จาก reference video...')
if PERSON_INDEX is not None:
    print(f'→ โหมด Multi-person: จะ track คนที่ {PERSON_INDEX} (นับจาก 0 = ซ้ายสุด)')

ref_data = extract_pose_from_video(
    ref_video_path,
    sample_fps=SAMPLE_FPS,
    person_index=PERSON_INDEX,   # None = คนเดียว, 0/1/2/... = เลือกคน
)
ref_data['video_info']['path'] = ref_video_path

os.makedirs(os.path.join(OUTPUT_DIR, 'pose_cache'), exist_ok=True)
save_pose_data(ref_data, os.path.join(OUTPUT_DIR, 'pose_cache', 'reference'))

print(f'\n✅ Extract สำเร็จ: {len(ref_data["keypoints"])} frames')
print(f'   Duration: {ref_data["video_info"]["duration"]:.1f}s')

กำลัง extract pose จาก reference video...


I0000 00:00:1776959426.496112  570275 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M3
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1776959426.532852  570277 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776959426.540284  570282 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.



วิดีโอ: [Mirrored] KARINA(카리나) - UP(업) 1인 커버댄스 ㅣ1인안무 거울모드.mp4
  Resolution: 1920x1080 | FPS: 30.0 | Duration: 164.1s
  โหมด: Single-person (MediaPipe เลือกเอง)


Extracting poses:   0%|                                | 0/2458 [00:00<?, ?it/s]W0000 00:00:1776959426.607564  570284 landmark_projection_calculator.cc:81] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.
Extracting poses: 2459it [00:45, 54.30it/s]                                     


✓ สกัดท่าทางสำเร็จ: 2456 frames (99.9% detection rate)
✓ บันทึก pose data: results/pose_cache/reference.npz

✅ Extract สำเร็จ: 2456 frames
   Duration: 164.1s


In [ ]:
# Cell 6: ดู sample poses ของ reference (optional)
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
fig.patch.set_facecolor('#1a1a2e')

kps_list = ref_data['keypoints']
sample_indices = np.linspace(0, len(kps_list)-1, 5, dtype=int)

for ax, idx in zip(axes, sample_indices):
    kp = kps_list[idx]
    ax.set_facecolor('#16213e')
    ax.scatter(kp[:, 0], -kp[:, 1], c='#00d4ff', s=30, zorder=5)
    
    # วาด connections
    connections = [(0,1),(0,2),(1,3),(2,4),(3,5),(4,6),(1,7),(2,8),(7,8),(7,9),(8,10),(9,11),(10,12)]
    for i, j in connections:
        if i < len(kp) and j < len(kp):
            ax.plot([kp[i,0], kp[j,0]], [-kp[i,1], -kp[j,1]], 
                   color='#ff6b9d', linewidth=1.5, alpha=0.8)
    
    ts = ref_data['timestamps'][idx]
    ax.set_title(f't={ts:.1f}s', color='white', fontsize=9)
    ax.axis('off')

plt.suptitle('Sample Poses - Reference (Artist)', color='white', fontsize=12)
plt.tight_layout()
plt.show()
print('👆 ท่าทางของศิลปินที่ extract ได้')

## 🕺 Step 3: เปรียบเทียบท่าเต้น

In [ ]:
# Cell 7: เปรียบเทียบท่าเต้น

# ถ้า USER_VIDEO เป็น YouTube URL → download ก่อน
user_video_path = USER_VIDEO
if USER_VIDEO and (USER_VIDEO.startswith("http") or "youtu" in USER_VIDEO):
    print("🔗 พบ YouTube URL → กำลัง download วิดีโอของคุณ...")
    user_video_path = download_youtube_video(
        url=USER_VIDEO,
        output_dir=os.path.join(OUTPUT_DIR, "user_videos"),
        start_time=USER_START_TIME,
        end_time=USER_END_TIME,
    )
    print(f"✅ download เสร็จ: {user_video_path}")

if not os.path.exists(user_video_path):
    print(f'❌ ไม่พบวิดีโอ: {user_video_path}')
    print('   กรุณาใส่ path หรือ YouTube URL ที่ถูกต้องใน USER_VIDEO')
else:
    results = compare_dance(
        reference_data=ref_data,
        user_video_path=user_video_path,
        output_dir=OUTPUT_DIR,
        sample_fps=SAMPLE_FPS,
        create_comparison_video=True,
    )
    print('✅ วิเคราะห์เสร็จแล้ว!')

## 📊 Step 4: ดูผลลัพธ์

In [ ]:
# Cell 8: แสดงกราฟวิเคราะห์
if 'results' in dir() and results.get('output_plot'):
    img = mpimg.imread(results['output_plot'])
    plt.figure(figsize=(14, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('กรุณารัน Cell 7 ก่อน')

In [ ]:
# Cell 9: Body Part Analysis Chart
if 'results' in dir():
    body = results['body_analysis']
    parts = list(body.keys())
    scores = [body[p]['score'] for p in parts]
    grades = [body[p]['grade'] for p in parts]

    colors = []
    for s in scores:
        if s >= 0.8:   colors.append('#00cc44')
        elif s >= 0.6: colors.append('#ffcc00')
        else:          colors.append('#ff4444')

    fig, ax = plt.subplots(figsize=(8, 5))
    fig.patch.set_facecolor('#1a1a2e')
    ax.set_facecolor('#16213e')

    bars = ax.barh(parts, scores, color=colors, height=0.5, edgecolor='none')

    for bar, score, grade in zip(bars, scores, grades):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f'{score:.0%} ({grade})', va='center', color='white', fontsize=11)

    ax.axvline(x=0.8, color='#00cc44', linestyle='--', alpha=0.5, linewidth=1)
    ax.axvline(x=0.6, color='#ffcc00', linestyle='--', alpha=0.5, linewidth=1)

    ax.set_xlim(0, 1.2)
    ax.set_title('Body Part Similarity', color='white', fontsize=13, pad=12)
    ax.tick_params(colors='#aaaaaa')
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_color('#444444')
    ax.spines['left'].set_color('#444444')
    for label in ax.get_yticklabels():
        label.set_color('white')

    plt.tight_layout()
    plt.show()

    # Feedback
    worst_part = min(body, key=lambda p: body[p]['score'])
    best_part = max(body, key=lambda p: body[p]['score'])
    print(f'\n💪 ส่วนที่ดีที่สุด:    {best_part}  ({body[best_part]["score"]:.0%})')
    print(f'🔧 ส่วนที่ต้องปรับปรุง: {worst_part} ({body[worst_part]["score"]:.0%})')
else:
    print('กรุณารัน Cell 7 ก่อน')

In [ ]:
# Cell 10: ดู Comparison Video (ถ้าสร้างสำเร็จ)
if 'results' in dir() and results.get('output_video') and os.path.exists(results['output_video']):
    print(f'Video saved: {results["output_video"]}')
    display(Video(results['output_video'], width=800, embed=True))
else:
    print('Comparison video ไม่ถูกสร้าง หรือกรุณารัน Cell 7 ก่อน')

---
## 📖 Bonus: สร้าง PDF คู่มือท่าเต้น

ภาพจริงจากวิดีโอ + skeleton วาดทับ รวมเป็น PDF พิมพ์ออกมาวางข้างกระจกตอนฝึกได้เลย!

In [6]:
# Cell 11: ตั้งค่า PDF Guide
from pose_guide_generator import generate_dance_guide

SONG_NAME  = "Up"   # 👈 แสดงบนหน้าปก PDF
MAX_POSES  = 400    # จำนวนเฟรมสูงสุด (filmstrip ได้เยอะ)
PDF_MODE   = "filmstrip"  # "hold" = หยุดนิ่ง | "motion" = ขยับ | "filmstrip" = ทุก 0.5 วิ
PDF_OUTPUT = os.path.join(OUTPUT_DIR, "dance_guide.pdf")

print(f'เพลง: {SONG_NAME}')
print(f'โหมด: {PDF_MODE}')
print(f'จำนวนเฟรมสูงสุด: {MAX_POSES}')
print(f'Output: {PDF_OUTPUT}')

เพลง: Up
โหมด: filmstrip
จำนวนเฟรมสูงสุด: 400
Output: results/dance_guide.pdf


In [7]:
# Cell 12: สร้าง PDF คู่มือท่าเต้น 🎉
pdf_path = generate_dance_guide(
    video_path=ref_video_path,
    output_path=PDF_OUTPUT,
    person_index=PERSON_INDEX,
    max_poses=MAX_POSES,
    poses_per_row=2,
    song_name=SONG_NAME,
    mode=PDF_MODE,
)

print(f'\n✅ PDF พร้อมแล้ว! เปิดได้ที่: {pdf_path}')
print('   → พิมพ์ออกมาวางข้างกระจกตอนฝึกได้เลยครับ 🎵')

I0000 00:00:1776959535.486251  571022 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M3
W0000 00:00:1776959535.549706  571023 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776959535.557879  571023 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


  📖 K-Pop Dance Guide Generator

[1/3] ตรวจจับท่าหลัก...

[Pose Detector] วิดีโอ: [Mirrored] KARINA(카리나) - UP(업) 1인 커버댄스 ㅣ1인안무 거울모드.mp4
  FPS: 30.0 | Sample: 10.0 fps | Threshold: 0.03


Scanning poses: 2459it [00:45, 54.23it/s]                                       


  ✓ เลือกได้ 400 ท่าต่อเนื่อง

[2/3] เตรียมภาพพร้อม skeleton...


Preparing images: 100%|███████████████████████| 400/400 [00:22<00:00, 17.70it/s]



[3/3] สร้าง PDF คู่มือ...

✅ สร้าง PDF สำเร็จ: results/dance_guide.pdf
   400 ท่า | 200 แถว

✅ PDF พร้อมแล้ว! เปิดได้ที่: results/dance_guide.pdf
   → พิมพ์ออกมาวางข้างกระจกตอนฝึกได้เลยครับ 🎵


---
## 🔄 วิธีใช้ซ้ำกับวิดีโอใหม่ (ไม่ต้อง extract reference ใหม่)

หลังจาก extract reference แล้วครั้งแรก สามารถโหลด cache และเปรียบเทียบกับวิดีโอใหม่ได้เลย

In [ ]:
# Cell 11: โหลด cached reference และเปรียบเทียบวิดีโอใหม่

# โหลด reference ที่ extract ไว้แล้ว
# ref_data_cached = load_pose_data(os.path.join(OUTPUT_DIR, 'pose_cache', 'reference'))

# เปรียบเทียบกับวิดีโอใหม่
# new_video = "my_dance_take2.mp4"
# results2 = compare_dance(ref_data_cached, new_video, output_dir=OUTPUT_DIR)

print('Uncomment โค้ดด้านบนเพื่อเปรียบเทียบกับวิดีโอใหม่')

---
## 💡 Tips สำหรับผลลัพธ์ที่ดีขึ้น

**เพิ่ม accuracy:**
- ใช้ `SAMPLE_FPS = 15` หรือ `20` เพื่อวิเคราะห์ละเอียดขึ้น (ช้าลงบ้าง)
- ตัดช่วงท่าเต้นที่ต้องการจาก YouTube ให้ตรงกัน

**ถ้า Detection Rate ต่ำ (<60%):**
- ตรวจสอบแสงในวิดีโอ
- ให้เห็นทั้งตัวในกล้อง
- ลด `min_detection_confidence` ในฟังก์ชัน `extract_pose_from_video`

**K-pop Songs ที่ทดสอบได้ดี:**
- เพลงที่มี Dance Practice (ห้องซ้อม) มักให้ผลดีที่สุด
- MV ที่กล้องไม่เปลี่ยนเร็วมาก
- ระวัง MV ที่มีหลายคนเต้นพร้อมกัน (จะ detect คนที่อยู่หน้าสุด)